# Zoom and resolution

## Level of detail

`zoom="auto"` (the default) draws the style at its size on a web page, so labels keep their size on paper. A number is contextily's zoom of 256-pixel tiles of the same scale, for the limits the axes have when the basemap is added, and `zoom_adjust` adds to either; both may be fractional.

In [ ]:
%pip install -q basemaplibre geopandas

In [ ]:
import geopandas
import matplotlib.pyplot as plt

import basemaplibre as bml

In [ ]:
def around(lon, lat, half_width):
    """Axes limits of a square around a point, half_width Web Mercator meters to each side."""
    x, y = geopandas.GeoSeries.from_xy([lon], [lat], crs=4326).to_crs(3857).iloc[0].coords[0]
    return x - half_width, x + half_width, y - half_width, y + half_width

In [ ]:
panels = {"zoom=13": {"zoom": 13}, 'zoom="auto"': {}, "zoom_adjust=1": {"zoom_adjust": 1}}
fig, axs = plt.subplots(1, 3, figsize=(9, 3.4), layout="constrained")
for ax, (title, options) in zip(axs, panels.items(), strict=True):
    ax.axis(around(21.013, 52.237, 1500))
    ax.set_aspect("equal")
    ax.set_axis_off()
    ax.set_title(title)
    bml.add_basemap(ax, source="OpenFreeMap.Positron", **options)

## Resolution

The basemap is rendered at the dpi the figure is drawn at, so `fig.savefig(..., dpi=300)` is sharp, but with 200 pixels to the inch at least, for HiDPI screens. `pixel_ratio`, the image's pixels per CSS pixel, fixes it instead (with `zoom="auto"` a CSS pixel is 1/96 inch, so the default is `max(200, dpi) / 96`): lower renders faster. Only the sharpness changes, not the map. Here, the Lake in Central Park at `pixel_ratio=1`, the default and `pixel_ratio=4`, and below them the framed detail of each image, enlarged to show its pixels.

In [ ]:
lake = around(-73.9712, 40.7762, 500)
x, y = (lake[0] + lake[1]) / 2, (lake[2] + lake[3]) / 2
detail = (x + 35, x + 315, y - 10, y + 270)  # Azalea Pond
fig, axs = plt.subplots(2, 3, figsize=(9, 6.6), layout="constrained")
for (ax, enlarged), ratio in zip(axs.T, [1, None, 4], strict=True):
    ax.axis(lake)
    ax.set_aspect("equal")
    ax.set_axis_off()
    name = "default: max(200, dpi) / 96" if ratio is None else f"pixel_ratio={ratio}"
    ax.set_title(f"{name}\n", fontsize=10)  # the size of the image goes on the second line
    bml.add_basemap(ax, source="OpenFreeMap.Liberty", pixel_ratio=ratio)
    image = ax.images[0].get_array()
    height, width = image.shape[:2]
    ax.set_title(f"{name}\n{width}×{height} pixels", fontsize=10)
    ax.indicate_inset(
        (detail[0], detail[2], detail[1] - detail[0], detail[3] - detail[2]), edgecolor="crimson"
    )
    enlarged.imshow(image, extent=lake, interpolation="nearest")
    enlarged.axis(detail)
    enlarged.set_axis_off()